In [1]:
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
from qiskit import QuantumCircuit, execute, Aer
from qiskit.providers.aer import QasmSimulator
import numpy as np
import struct

global ciphertext_bits, encryption_key
ciphertext_bits = ""
encryption_key = ""

def alice_state_prep(state, basis):
    num_qubits = len(state)
    circuit = QuantumCircuit(num_qubits)
    for i in range(len(basis)):
        if state[i] == 1:
            circuit.x(i)
        if basis[i] == 1:
            circuit.h(i)
    return circuit

def bob_measurement(circuit, basis):
    for i in range(len(basis)):
        if basis[i] == 1:
            circuit.h(i)
    circuit.measure_all()

def key_creation(circuit, alice_basis, bob_basis):
    key = execute(circuit.reverse_bits(), backend=QasmSimulator(), shots=1).result().get_counts().most_frequent()
    encryption_key = ''
    for i in range(len(alice_basis)):
        if alice_basis[i] == bob_basis[i]:
            encryption_key += str(key[i])
    return encryption_key

def point_to_bits(triplet):
    return ''.join(format(struct.unpack('!Q', struct.pack('!d', t))[0], '064b') for t in triplet)

def encrypt_message(_):
    global ciphertext_bits, encryption_key
    try:
        triplet = tuple(map(float, triplet_input.value.split(',')))
    except ValueError:
        with output:
            clear_output()
            print("Invalid input. Please enter three comma-separated numbers.")
        return
    
    plain_text_bits = point_to_bits(triplet)
    num_qubits = int(2.0*len(plain_text_bits))
    alice_basis = np.random.randint(2, size=num_qubits)
    alice_state = np.random.randint(2, size=num_qubits)
    cir = alice_state_prep(alice_state, alice_basis)
    bob_basis = np.random.randint(2, size=num_qubits)
    bob_measurement(cir, bob_basis)
    encryption_key = key_creation(cir, alice_basis, bob_basis)
    encryption_key = encryption_key[:len(plain_text_bits)]
    ciphertext_bits = ''.join(str(int(plain_text_bits[i]) ^ int(encryption_key[i])) for i in range(len(plain_text_bits)))
    
    clear_output()
    display(triplet_input, encrypt_button, decrypt_button, output)
    with output:
        print("Ciphertext (in binary):", ciphertext_bits)
        print("Key:", encryption_key)

def binary_to_float(b):
    int_value = int(b, 2)
    return struct.unpack('!d', struct.pack('!Q', int_value))[0]

def decrypt_message(_):
    global ciphertext_bits, encryption_key
    if not ciphertext_bits or not encryption_key:
        with output:
            clear_output()
            print("No encrypted message found. Please encrypt a message first.")
        return
    
    decrypted_bits = ''.join(str(int(ciphertext_bits[i]) ^ int(encryption_key[i])) for i in range(len(ciphertext_bits)))
    triplet_bits = [decrypted_bits[i:i + 64] for i in range(0, len(decrypted_bits), 64)]
    original_triplet = tuple(binary_to_float(bit) for bit in triplet_bits)
    
    clear_output()
    display(triplet_input, encrypt_button, decrypt_button, output)
    with output:
        print("Decrypted triplet:", original_triplet)

triplet_input = widgets.Text(description="Enter triplet:", placeholder="e.g., 35.7602,-78.188,4000")
encrypt_button = widgets.Button(description="Encrypt Message")
decrypt_button = widgets.Button(description="Decrypt Message")
encrypt_button.on_click(encrypt_message)
decrypt_button.on_click(decrypt_message)
output = widgets.Output()

display(triplet_input, encrypt_button, decrypt_button, output)


Text(value='', description='Enter triplet:', placeholder='e.g., 35.7602,-78.188,4000')

Button(description='Encrypt Message', style=ButtonStyle())

Button(description='Decrypt Message', style=ButtonStyle())

Output()